In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.lines import Line2D
import scienceplots
plt.style.use(['science', 'nature', 'grid'])

import numpy as np
from scipy.interpolate import interp1d
import pandas as pd

from LoadMoments import *
from LoadTSV import *

In [ ]:
# mpl.use('pgf')
mpl.rcParams.update({
    "pgf.texsystem": "pdflatex",
    "text.usetex": True,
    "font.family": "serif",
    "font.size": 10,        # Match your LaTeX document font size
    "axes.titlesize": 10,
    "axes.labelsize": 10,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8
})

In [ ]:
# Plotting
colors = ['blue', 'orange', 'green', 'red', 'purple', 'brown']
linestyles = ['-', '--', '-.', (0, (3, 1, 1, 1)), (0, (5, 1))]

# Parameter
M = 4
closure = "ExtGram"
source_string = "relaxation_source"
T_end = 0.3
base_tree_level = 10
polydeg = 1
rho_L = 7.0
v_L1 = 0.0
v_L2 = 0.0
v_L3 = 0.0
theta_L = 1.0
rho_R = 1.0
v_R1 = 0.0
v_R2 = 0.0
v_R3 = 0.0
theta_R = 1.0

marker = ['o', 's', '^', 'x', 'd']

# Kn and relaxation for solution files
M_vec = [4, 6, 8]
source = 'relaxation_source'

In [ ]:
name_slab = lambda M, Kn, source_string: f"../out/1D3V/1D3V_angles/gram_solution_M{M}_closureExtGram_Kn{Kn}_source{source_string}_anglepairMax_rhoL7.0_cons.tsv"

name_full = lambda M, Kn, source_string: f"../out/1D3V_full/1D3V_M{M}_closure{closure}_Kn{Kn}_source{source_string}_angleindex1_cons.tsv"

In [ ]:
Moments_vec = [4, 6, 8]

for Kn in [0.2, 1.0]:
    for M in Moments_vec:
        fig, axs = plt.subplots(1, 2, figsize=(6.8, 2.5), sharex=True, sharey=True)
        time_query = 21 

        data_slab = read_solution_file(name_slab(M, Kn, source))
        times_slab = np.array([data_slab[ts]["t"] for ts in sorted(data_slab.keys())])
        timesteps_slab = np.array(sorted(data_slab.keys()))
        sol_slab = interpolate_solution(data_slab, time_query, times_slab, timesteps_slab)
        x_slab, U_slab = sol_slab["x"], sol_slab["u"]
        rho_slab = U_slab[:, 0]
        v_slab = U_slab[:, 1] / U_slab[:, 0] # vx
        theta_slab = 1 / (3 * rho_slab) * (U_slab[:, 2] + 2 * U_slab[:, 3] - rho_slab * v_slab**2) 
        p_slab = rho_slab * theta_slab

        data_full = read_solution_file(name_full(M, Kn, source))
        times_full = np.array([data_full[ts]["t"] for ts in sorted(data_full.keys())])
        timesteps_full = np.array(sorted(data_full.keys()))
        sol_full = interpolate_solution(data_full, time_query, times_full, timesteps_full)
        x_full, U_full = sol_full["x"], sol_full["u"]
        rho_full = U_full[:, 0]
        v_full = U_full[:, 1] / U_full[:, 0] # vx
        theta_full = 1 / (3 * rho_full) * (U_full[:, 4] + U_full[:, 7] + U_full[:, 9] - rho_full * v_full**2) 
        p_full = rho_full * theta_full

        axs[0].plot(x_slab, rho_slab, color='blue', label=r'$\rho$ (slab)', linestyle='-')
        axs[0].plot(x_full, rho_full, color='blue', label=r'$\rho$ (full)', linestyle='--')
        axs[0].set_ylabel(r'$\rho$')

        twx = axs[0].twinx()
        twx.plot(x_slab, v_slab, color='orange', label=r'$v$ (slab)', linestyle='-')
        twx.plot(x_full, v_full, color='orange', label=r'$v$ (full)', linestyle='--')
        twx.set_ylabel(r'$v$')

        axs[1].plot(x_slab, p_slab, color='green', label=r'$p$ (slab)', linestyle='-')
        axs[1].plot(x_full, p_full, color='green', label=r'$p$ (full)', linestyle='--')
        axs[1].set_ylabel(r'$p$')

        twx2 = axs[1].twinx()
        twx2.plot(x_slab, theta_slab, color='red', label=r'$\theta$ (slab)', linestyle='-')
        twx2.plot(x_full, theta_full, color='red', label=r'$\theta$ (full)', linestyle='--')
        twx2.set_ylabel(r'$\theta$')

        [ax.set_xlim(-1, 1) for ax in axs.flatten()]
        [ax.legend() for ax in axs.flatten()]

        fig.suptitle(f"Comparison of slab and full solutions for M={M}, Kn={Kn}")
        fig.tight_layout(); fig.show()